# HW4: Yield Curves from Instantaneous Forward Rates

Build a two-component model directly from IFR inputs, then query discount factors.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfAddPeriod,
    qfAccrued,
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfDiscountFactor,
)
from fixedincomelib.date import Date

## Remember try to implement all the `TODO` in the library.

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells. The funding component adds an IFR spread over SOFR.

In [2]:
value_date = "2026-10-07"
payment_currency = "USD"  # Placeholder; currently ignored by the model.
query_tenors = ["6M", "1Y", "2Y", "5Y", "10Y", "15Y"]

projection_ifr = pd.DataFrame(
    {
        "values": [
            0.030, 0.031, 0.032, 0.033, 0.034,
            0.035, 0.036, 0.037, 0.038, 0.039,
        ]
    },
    index=pd.Index([f"{year}Y" for year in range(1, 11)], name="Tenor"),
)

funding_ifr = pd.DataFrame(
    {"values": [0.0]},
    index=pd.Index(["1Y"], name="Tenor"),
)

display(projection_ifr.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_ifr.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Tenor,
1Y,0.030
2Y,0.031
3Y,0.032
4Y,0.033
5Y,0.034
6Y,0.035
7Y,0.036
8Y,0.037
9Y,0.038


,SOFR-1B-FLAT IFR spread
Tenor,
1Y,0.0


## 2. Data and build methods

The registered IFR conventions identify each dataset. Each build method maps its dataset to a target component.

In [3]:
projection_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE",
    "SOFR-1B-IFR",
    projection_ifr,
)

funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE",
    "SOFR-1B-FUNDING-IFR",
    funding_ifr,
)

data_collection = qfCreateDataCollection([projection_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
projection_bm = qfCreateBuildMethod(
    "YIELD_CURVE_INDEX",
    {
        "TARGET": "SOFR-1B",
        "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
    },
)

funding_bm = qfCreateBuildMethod(
    "YIELD_CURVE_FUNDING",
    {
        "TARGET": "SOFR-1B-FLAT",
        "REFERENCE INDEX":"SOFR-1B",
        "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
    },
)

build_methods = qfCreateModelBuildMethodCollection([projection_bm, funding_bm])

## 3. Build the model

Trivial calibration copies IFR values into state. Tenors use Following/USGS; curve times use ACT/ACT (ISDA). Defaults are piecewise-constant left-continuous interpolation and flat extrapolation.

The registered reference is `SOFR-1B-FLAT → SOFR-1B`.

In [5]:
model = qfCreateModel(
    value_date,
    "YIELD_CURVE",
    data_collection,
    build_methods,
)

projection_component = model.retrieve_model_component(projection_bm.target_index)
funding_component = model.retrieve_model_component(funding_bm.target_index)

state_table = pd.concat(
    [
        pd.DataFrame({
            "Component": projection_bm.target,
            "Tenor": projection_data.axis1,
            "Time (years)": projection_component.state_data[0],
            "IFR": projection_component.state_data[1],
        }),
        pd.DataFrame({
            "Component": funding_bm.target,
            "Tenor": funding_data.axis1,
            "Time (years)": funding_component.state_data[0],
            "IFR": funding_component.state_data[1],
        }),
    ],
    ignore_index=True,
)

display(state_table)

,Component,Tenor,Time (years),IFR
0,SOFR-1B,1Y,1.000000,0.030
1,SOFR-1B,2Y,2.008840,0.031
2,SOFR-1B,3Y,3.005479,0.032
3,SOFR-1B,4Y,4.000000,0.033
4,SOFR-1B,5Y,5.000000,0.034
5,SOFR-1B,6Y,6.000644,0.035
6,SOFR-1B,7Y,7.000000,0.036
7,SOFR-1B,8Y,8.008219,0.037
8,SOFR-1B,9Y,9.005479,0.038
9,SOFR-1B,10Y,10.000644,0.039


## 4. Discount factors

Each component returns $D_{\mathrm{local}}(t)=\exp(-\int_0^t f(s)\,ds)$. The model follows references and multiplies their factors:

$$D_{\mathrm{funding}}(t)=D_{\mathrm{SOFR}}(t)\,D_{\mathrm{funding,\ local}}(t).$$

With zero funding IFR, the local factor is 1 and the two complete curves agree.

In [6]:
query_dates = [value_date] + [
    qfAddPeriod(value_date, tenor, "F", "USGS")
    for tenor in query_tenors
]

discount_factors = pd.DataFrame({
    "Tenor": ["Value date"] + query_tenors,
    "Date": query_dates,
    "SOFR-1B": [
        qfDiscountFactor(model, "SOFR-1B", date, payment_currency)
        for date in query_dates
    ],
    "Funding component only": [
        funding_component.discount_factor(Date(date))
        for date in query_dates
    ],
    "SOFR-1B-FLAT": [
        qfDiscountFactor(model, "SOFR-1B-FLAT", date, payment_currency)
        for date in query_dates
    ],
})

display(discount_factors.round(8))

,Tenor,Date,SOFR-1B,Funding component only,SOFR-1B-FLAT
0,Value date,2026-10-07,1.000000,1.0,1.000000
1,6M,2027-04-07,0.985152,1.0,0.985152
2,1Y,2027-10-07,0.970446,1.0,0.970446
3,2Y,2028-10-10,0.940565,1.0,0.940565
4,5Y,2031-10-07,0.852156,1.0,0.852156
5,10Y,2036-10-07,0.708223,1.0,0.708223
6,15Y,2041-10-07,0.582765,1.0,0.582765


## 5. Cashflow price and analytic risk

### 5.1 Create product

Create a `ProductFixedAccruedCashflow` using the inputs below.

**Hint:** Use `qfCreateProductFixedAccruedCashflow`; set the payment date to the termination date.

In [ ]:
from fixedincomelib import qfCreateProductFixedAccruedCashflow

# Product inputs: edit these and rerun all three cells.
product_tenor = "5Y"
product_notional = 100.0
product_accrual_basis = "ACT/360"
termination_date = qfAddPeriod(value_date, product_tenor, "F", "USGS")

product = qfCreateProductFixedAccruedCashflow(
    #TODO
)

### 5.2 Price

Discount the product's payment using `SOFR-1B-FLAT`.

**Hint:** `product.accrued` is a year fraction. The fixed payment is $c=\text{product.notional}\times\text{product.accrued}$, so $V=c\,D_{\mathrm{SOFR-1B-FLAT}}(T)$.

In [ ]:
c = product.notional * product.accrued
df = qfDiscountFactor(
    #TODO
)
V = c * df

price_table = pd.DataFrame([{
    "Payment date": product.payment_date.ISO(),
    "Cashflow (c)": c,
    "Discount factor": df,
    "Price (V)": V,
}])
display(price_table.round(8))

,Payment date,Cashflow (c),Discount factor,Price (V)
0,2031-10-07,507.222222,0.852156,432.232456


### 5.3 Analytic risk

Compute $\nabla_{X^I}V$ for both components' IFR node values, holding node times and $c$ fixed. Use analytic derivatives, without bumping. Report derivatives per unit decimal IFR, without 1 bp scaling.

**Hint:** Use ACT/ACT (ISDA) for curve time $T$. The interpolator's `gradient_of_integrated_value_wrt_ordinate(0.0, T)` gives $w_i=\partial(\int_0^T f(s)\,ds)/\partial f_i$. Apply the chain rule to the full price:

$$\frac{\partial V}{\partial f_i}=-V\,w_i.$$

Include the funding node even when its IFR is zero; its derivative is $-TV$ for this single-node flat component.

In [ ]:
# Curve time uses the model's day count, not the product's accrual basis.
T = qfAccrued(
    model.value_date.ISO(), product.payment_date.ISO(), "ACT/ACT (ISDA)"
)

projection_interpolator = #TODO
funding_interpolator = #TODO
projection_weights = #TODO
funding_weights = #TODO

# Chain rule: V already includes both SOFR and funding discount factors.
projection_risk = #TODO
funding_risk = #TODO

# Parameter order: SOFR nodes, then funding nodes.
X_I = np.concatenate([
    projection_component.state_data[1], funding_component.state_data[1]
])
risk = np.concatenate([projection_risk, funding_risk])

risk_table = pd.DataFrame({
    "Component": (
        [projection_bm.target] * len(projection_risk)
        + [funding_bm.target] * len(funding_risk)
    ),
    "Tenor": projection_data.axis1 + funding_data.axis1,
    "IFR (X_I)": X_I,
    "dV/dX_I": risk,
})
display(risk_table.round(8))

,Component,Tenor,IFR (X_I),dV/dX_I
0,SOFR-1B,1Y,0.030,-432.232456
1,SOFR-1B,2Y,0.031,-436.053599
2,SOFR-1B,3Y,0.032,-430.779710
3,SOFR-1B,4Y,0.033,-429.864059
4,SOFR-1B,5Y,0.034,-432.232456
5,SOFR-1B,6Y,0.035,-0.000000
6,SOFR-1B,7Y,0.036,-0.000000
7,SOFR-1B,8Y,0.037,-0.000000
8,SOFR-1B,9Y,0.038,-0.000000
9,SOFR-1B,10Y,0.039,-0.000000
